Notebook 2. Proyección ortogonal sobre $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$ en $\mathbb{R}^3$
=====================================================================================================

**Author:** Marcos Bujosa



<div class="abstract" id="orgbbf6e3b">
<p>
Visualización interactiva en $\mathbb{R}^3$ de la regresión lineal simple (lección 7): el plano $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$, el ajuste $\boldsymbol{\mathop{\widehat{y}}}$ y su residuo $\boldsymbol{\mathop{\widehat{e}}}$, la identidad que relaciona los vectores en desviaciones del ajuste y del regresor, y qué le exigimos exactamente al vector generador $\boldsymbol{u}$ para que MCO recupere los coeficientes verdaderos. Complemento de las lecciones 6 y 7.
</p>

</div>

-   ([mybinder](https://mybinder.org/v2/gh/mbujosab/PEconometria/gh-pages?labpath=CuadernosElectronicos/S09-Notebook02.ipynb))



## Introducción



Este cuaderno retoma la construcción del primer cuaderno (proyección sobre $\mathcal{L}(\boldsymbol{1})$, lección 4) y añade la pieza que introducen las lecciones 6 y 7: un *segundo* vector generador, $\boldsymbol{x}$. Ya no proyectamos sobre una recta, sino sobre el *plano* $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$.

Seguimos trabajando en $\mathbb{R}^3$ ($n=3$ observaciones) para poder dibujar la escena, pero conviene señalar desde el principio una diferencia geométrica importante respecto al primer cuaderno: allí, con un solo generador $\boldsymbol{1}$, el complemento ortogonal $\mathcal{L}(\boldsymbol{1})^\perp$ era un *plano* (dimensión 2) dentro de $\mathbb{R}^3$. Aquí, con *dos* generadores ($\boldsymbol{1}$ y $\boldsymbol{x}$), el complemento ortogonal $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})^\perp$ es una *recta* (dimensión 1): solo queda una dirección posible para el residuo. Esta observación será la clave de la Parte 3.

Podrás, como en el primer cuaderno, rotar las figuras con el ratón y editar los vectores en las celdas de código para crear tus propios ejemplos. Además, junto a cada figura 3D verás su gemela: el diagrama de dispersión clásico $(x_i,y_i)$ con la recta ajustada, para no perder de vista la lectura “de manual” de lo mismo.



## Herramientas auxiliares



#### Módulos y configuración



Como en el primer cuaderno, usamos `numpy` y `plotly` para la geometría en $\mathbb{R}^3$; añadimos `matplotlib` para el diagrama de dispersión clásico (estático: aquí no hay ningún ángulo que rotar).



In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt

pio.renderers.default = "notebook"

#### El ajuste MCO de la regresión simple (lección 7)



Esta es la única pieza puramente algebraica del cuaderno: las fórmulas de $\hat\beta_1$ y $\hat\beta_2$ obtenidas en la lección 7 a partir de las dos condiciones de ortogonalidad. Añadimos también, como en el primer cuaderno, el vector de medias y el vector en desviaciones (lección 4), que necesitaremos para las Partes 2 y 3.



In [1]:
def ajuste_mco_simple(y, x):
    """Ajuste MCO de la regresión simple y = beta0 + beta1*x (lección 7).
    Devuelve (yhat, ehat, beta0, beta1)."""
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    mu_x = x.mean()
    mu_y = y.mean()
    sigma_x2 = np.mean((x - mu_x) ** 2)
    sigma_xy = np.mean((x - mu_x) * (y - mu_y))
    beta1 = sigma_xy / sigma_x2
    beta0 = mu_y - beta1 * mu_x
    yhat = beta0 + beta1 * x
    ehat = y - yhat
    return yhat, ehat, beta0, beta1

def vector_de_medias(v):
    """Devuelve (vector_de_medias, mu): el vector constante mu*1 y el
    escalar mu = media aritmética de las componentes de v."""
    v = np.asarray(v, dtype=float)
    mu = v.mean()
    return np.full(v.shape, mu), mu

def vector_en_desviaciones(v):
    """Devuelve (v - vector_de_medias(v), mu)."""
    v = np.asarray(v, dtype=float)
    vbar, mu = vector_de_medias(v)
    return v - vbar, mu

#### Bases ortonormales: el plano fijo $\mathcal{L}(\boldsymbol{1})^\perp$ y el plano $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$



Necesitamos dos construcciones distintas. La primera, ya usada en el primer cuaderno, es una base ortonormal *fija* del plano $\mathcal{L}(\boldsymbol{1})^\perp$ (no depende de $\boldsymbol{x}$; la reutilizaremos en la Parte 2).

La segunda es la base del plano $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$, y aquí reciclamos directamente lo visto en la lección 6: como $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})=\mathcal{L}(\boldsymbol{1},\boldsymbol{x}-\boldsymbol{\mathop{\overline{x}}})$, y el vector en desviaciones $\boldsymbol{x}-\boldsymbol{\mathop{\overline{x}}}$ es *automáticamente* ortogonal a $\boldsymbol{1}$ (tiene media cero, lección 4), no necesitamos ningún truco adicional: basta normalizar cada uno de estos dos generadores ya ortogonales entre sí.



In [1]:
def base_plano_ortogonal():
    """Base ortonormal (u1, u2) del plano L(1)^perp en R^3 (fija)."""
    uno = np.array([1., 1., 1.]) / np.sqrt(3)
    u1 = np.array([1., -1., 0.])
    u1 = u1 / np.linalg.norm(u1)
    u2 = np.cross(uno, u1)
    u2 = u2 / np.linalg.norm(u2)
    return u1, u2

def base_ortonormal_L1x(x):
    """Base ortonormal (u1, u2) del plano L(1,x), reutilizando la lección 6:
    L(1,x) = L(1, x - x_barra), y x - x_barra ya es ortogonal a 1 (media
    cero). Basta normalizar cada generador; no hace falta Gram-Schmidt."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    u1 = np.ones(n) / np.sqrt(n)
    ex, _ = vector_en_desviaciones(x)
    u2 = ex / np.linalg.norm(ex)
    return u1, u2

#### Funciones de dibujo



Reutilizamos (redefiniéndolas, pues este cuaderno es un fichero independiente) las funciones del primer cuaderno para dibujar $\mathcal{L}(\boldsymbol{1})$ con sus marcas graduadas y para colocar vectores y segmentos. Añadimos varias piezas nuevas: un eje genérico con sus propias marcas graduadas (para dibujar $\mathcal{L}(\boldsymbol{x})$, el segundo generador del plano, y poder "contar" pasos de longitud $\|\boldsymbol{x}\|$ al desplazarse por él), el plano $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$ (usando la base de la lección 6), y una función que dibuja las dos componentes del ajuste, $\hat\beta_1\boldsymbol{1}$ y $\hat\beta_2\boldsymbol{x}$, junto con el paralelogramo que las suma en $\boldsymbol{\mathop{\widehat{y}}}$ (la misma idea que en la figura de las lecciones 6-7, ahora interactiva).



In [1]:
def marcas_unitarias_L1(fig, rango=4, color='blue'):
    """Marcas sobre L(1), una por cada valor entero de la media."""
    uno = np.array([1., 1., 1.])
    ks = np.arange(-rango, rango + 1)
    puntos = np.outer(ks, uno)
    fig.add_trace(go.Scatter3d(
        x=puntos[:, 0], y=puntos[:, 1], z=puntos[:, 2],
        mode='markers',
        marker=dict(size=4, color=color, symbol='cross'),
        text=[f"mu = {k}" for k in ks],
        hoverinfo='text', showlegend=False))

def marcas_eje(fig, v, rango=4, color='black', etiqueta='k'):
    """Marcas sobre la recta L(v), una por cada múltiplo entero de v (el
    análogo, para un vector genérico v, de marcas_unitarias_L1): permiten
    'contar', desplazándose sobre la recta, cuántas veces hay que sumar
    (o restar) v para llegar a un punto dado de esa recta."""
    v = np.asarray(v, dtype=float)
    norma_v = np.linalg.norm(v)
    if norma_v < 1e-12:
        return
    kmax = int(np.floor(rango / norma_v + 1e-9))
    ks = np.arange(-kmax, kmax + 1)
    puntos = np.outer(ks, v)
    fig.add_trace(go.Scatter3d(
        x=puntos[:, 0], y=puntos[:, 1], z=puntos[:, 2],
        mode='markers',
        marker=dict(size=4, color=color, symbol='cross'),
        text=[f"{etiqueta} = {k}" for k in ks],
        hoverinfo='text', showlegend=False))

def marcas_desviacion(fig, base, e, n, color, etiqueta='sigma'):
    """Marcas sobre el segmento base -> base+e, espaciadas de modo que
    cada marca sucesiva corresponde a un incremento de 1 en sigma (norma
    ESTADÍSTICA de e); ver primer cuaderno para el detalle geométrico."""
    norma_euclidea_e = np.linalg.norm(e)
    if norma_euclidea_e < 1e-12:
        return
    sigma = norma_euclidea_e / np.sqrt(n)
    u_e = e / norma_euclidea_e
    paso_euclideo = np.sqrt(n)
    kmax = int(np.floor(sigma + 1e-9))
    ks = np.arange(0, kmax + 1)
    puntos = base + np.outer(ks, u_e * paso_euclideo)
    fig.add_trace(go.Scatter3d(
        x=puntos[:, 0], y=puntos[:, 1], z=puntos[:, 2],
        mode='markers',
        marker=dict(size=4, color=color, symbol='cross'),
        text=[f"{etiqueta} = {k}" for k in ks],
        hoverinfo='text', showlegend=False))

def recta_L1(fig, rango=4, color='blue'):
    """Dibuja L(1) con sus marcas graduadas (una por unidad de mu)."""
    t = np.linspace(-rango, rango, 2)
    fig.add_trace(go.Scatter3d(
        x=t, y=t, z=t, mode='lines',
        line=dict(color=color, width=5), name='L(1)'))
    marcas_unitarias_L1(fig, rango=rango, color=color)

def eje_generado(fig, v, rango=4, color='black', nombre=''):
    """Dibuja la recta L(v) que pasa por el origen en la dirección de v
    (sin marcas graduadas por defecto: ver marcas_eje para añadirlas)."""
    v = np.asarray(v, dtype=float)
    v_unit = v / np.linalg.norm(v)
    t = np.linspace(-rango, rango, 2)
    pts = np.outer(t, v_unit)
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='lines', line=dict(color=color, width=3),
        name=nombre, hoverinfo='skip', showlegend=True))

def plano_ortogonal(fig, rango=4, color='lightslategray'):
    """Parche + rejilla de L(1)^perp (fijo)."""
    u1, u2 = base_plano_ortogonal()
    S, T = np.meshgrid([-rango, rango], [-rango, rango])
    X = S * u1[0] + T * u2[0]
    Y = S * u1[1] + T * u2[1]
    Z = S * u1[2] + T * u2[2]
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z, showscale=False, opacity=0.12,
        colorscale=[[0, color], [1, color]], hoverinfo='skip'))
    for s in range(-rango, rango + 1):
        p0 = s * u1 - rango * u2
        p1 = s * u1 + rango * u2
        fig.add_trace(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode='lines', line=dict(color='gray', width=1),
            showlegend=False, hoverinfo='skip'))
    for t in range(-rango, rango + 1):
        p0 = -rango * u1 + t * u2
        p1 = rango * u1 + t * u2
        fig.add_trace(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode='lines', line=dict(color='gray', width=1),
            showlegend=False, hoverinfo='skip'))

def plano_L1x(fig, x, rango=4, color='lightslategray'):
    """Parche + rejilla de L(1,x), usando la base ortonormal de la lección
    6 (1 y el vector en desviaciones de x)."""
    u1, u2 = base_ortonormal_L1x(x)
    S, T = np.meshgrid([-rango, rango], [-rango, rango])
    X = S * u1[0] + T * u2[0]
    Y = S * u1[1] + T * u2[1]
    Z = S * u1[2] + T * u2[2]
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z, showscale=False, opacity=0.12,
        colorscale=[[0, color], [1, color]], hoverinfo='skip'))
    for s in range(-rango, rango + 1):
        p0 = s * u1 - rango * u2
        p1 = s * u1 + rango * u2
        fig.add_trace(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode='lines', line=dict(color='gray', width=1),
            showlegend=False, hoverinfo='skip'))
    for t in range(-rango, rango + 1):
        p0 = -rango * u1 + t * u2
        p1 = rango * u1 + t * u2
        fig.add_trace(go.Scatter3d(
            x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
            mode='lines', line=dict(color='gray', width=1),
            showlegend=False, hoverinfo='skip'))


def agrega_vector(fig, punto, color='crimson', nombre='v', dash='solid', width=6):
    """Vector desde el origen hasta 'punto' (línea + marcador en la punta)."""
    punto = np.asarray(punto, dtype=float)
    fig.add_trace(go.Scatter3d(
        x=[0, punto[0]], y=[0, punto[1]], z=[0, punto[2]],
        mode='lines+markers',
        line=dict(color=color, width=width, dash=dash),
        marker=dict(size=[0, 5], color=color), name=nombre))

def agrega_segmento(fig, p0, p1, color='gray', width=3, dash='dot'):
    """Segmento discontinuo entre dos puntos."""
    fig.add_trace(go.Scatter3d(
        x=[p0[0], p1[0]], y=[p0[1], p1[1]], z=[p0[2], p1[2]],
        mode='lines', line=dict(color=color, width=width, dash=dash),
        showlegend=False, hoverinfo='skip'))

def agrega_componentes_ajuste(fig, beta0, beta1, x, yhat):
    """Dibuja beta0*1 y beta1*x, y los segmentos punteados del
    paralelogramo que, sumados, reconstruyen yhat."""
    x = np.asarray(x, dtype=float)
    uno = np.ones_like(x)
    comp0 = beta0 * uno
    comp1 = beta1 * x
    agrega_vector(fig, comp0, color='darkorange', nombre='beta0*1', dash='dot', width=10)
    agrega_vector(fig, comp1, color='sienna', nombre='beta1*x', dash='dot', width=10)
    agrega_segmento(fig, comp0, yhat, color='dimgray', width=5, dash='dash')
    agrega_segmento(fig, comp1, yhat, color='dimgray', width=5, dash='dash')

#### Las dos figuras compuestas: el ajuste en $\mathbb{R}^3$ y su gemela en 2D



La primera función compone toda la escena del ajuste: el plano $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$ (con sus dos ejes, oblicuos entre sí), el vector de datos $\boldsymbol{y}$, el ajuste $\boldsymbol{\mathop{\widehat{y}}}$, el residuo $\boldsymbol{\mathop{\widehat{e}}}$ (con su regla graduada, como en el primer cuaderno) y, opcionalmente, las dos componentes $\hat\beta_1\boldsymbol{1}$ y $\hat\beta_2\boldsymbol{x}$. La segunda es su gemela clásica: el diagrama de dispersión $(x_i,y_i)$ con la recta ajustada $\hat y=\hat\beta_1+\hat\beta_2 x$.



In [1]:
def figura_ajuste_R3(x, y, rango=5, titulo="", mostrar_componentes=True):
    """Compone la escena: L(1), el eje L(x) con sus marcas, el plano
    L(1,x), el vector y, el ajuste yhat, el residuo ehat y (opcionalmente)
    sus dos componentes sobre los generadores del plano."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    yhat, ehat, beta0, beta1 = ajuste_mco_simple(y, x)

    fig = go.Figure()
    recta_L1(fig, rango=rango)
    eje_generado(fig, x, rango=rango, color='black', nombre='L(x)')
    marcas_eje(fig, x, rango=rango, color='black', etiqueta='k (punto = k*x)')
    plano_L1x(fig, x, rango=rango)

    agrega_vector(fig, y, color='seagreen', nombre='y')
    agrega_vector(fig, yhat, color='crimson', nombre='yhat')
    agrega_segmento(fig, yhat, y, color='gray')
    marcas_desviacion(fig, yhat, ehat, n, color='gray', etiqueta='sigma_e')

    if mostrar_componentes:
        agrega_componentes_ajuste(fig, beta0, beta1, x, yhat)

    fondo = dict(showbackground=True, backgroundcolor='rgb(235,235,245)',
                 gridcolor='white', dtick=1, range=[-rango, rango])
    fig.update_layout(
        scene=dict(xaxis=fondo, yaxis=fondo, zaxis=fondo, aspectmode='cube'),
        title=titulo, width=800, height=700,
        margin=dict(l=0, r=0, b=0, t=40))
    return fig

def diagrama_dispersion(x, y, mostrar_residuos=False, titulo=""):
    """Diagrama de dispersión clásico, con la recta MCO ajustada y,
    opcionalmente, los residuos como segmentos verticales discontinuos."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    yhat, ehat, beta0, beta1 = ajuste_mco_simple(y, x)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(x, y, color='seagreen', zorder=3, label='datos (x_i, y_i)')
    xs = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)
    ax.plot(xs, beta0 + beta1 * xs, color='crimson',
            label=f"yhat = {beta0:.2f} + {beta1:.2f}*x")
    if mostrar_residuos:
        for xi, yi, yhi in zip(x, y, yhat):
            ax.plot([xi, xi], [yi, yhi], color='gray',
                    linestyle='dotted', zorder=2)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(titulo)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

#### Ejemplos para explorar



Un pequeño diccionario de pares $(\boldsymbol{x},\boldsymbol{y})$ mejor o peor ajustados: `canonico` es el ejemplo de la lección 5 ($\rho_{\boldsymbol{x}\boldsymbol{y}}=\sqrt3/2$); `fuerte` tiene una correlación muy alta; `debil`, muy baja. Cámbialos, o añade el tuyo, para repetir cualquier actividad del cuaderno con otros datos.



In [1]:
ejemplos = {
    'canonico': (np.array([1., 2., 3.]), np.array([2., 2., 5.])),
    'fuerte':   (np.array([1., 2., 3.]), np.array([2., 5., 9.])),
    'debil':    (np.array([1., 2., 3.]), np.array([5., 1., 6.])),
}

## Parte 1: el ajuste y su residuo



Elegimos un ejemplo y dibujamos la escena completa: el plano $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})$ (nota, al rotar la figura, que sus dos ejes —$\mathcal{L}(\boldsymbol{1})$ en azul y $\mathcal{L}(\boldsymbol{x})$ en negro— son oblicuos, no perpendiculares, exactamente como advertía la lección 6), el vector de datos $\boldsymbol{y}$, su ajuste $\boldsymbol{\mathop{\widehat{y}}}$, y el residuo $\boldsymbol{\mathop{\widehat{e}}}$ como el segmento perpendicular al plano que los conecta. Fíjate en las marcas negras sobre $\mathcal{L}(\boldsymbol{x})$: cada una indica, al pasar el ratón por encima, cuántas veces $\boldsymbol{x}$ hay que sumar (con signo) para llegar a ese punto de la recta —igual que las marcas azules sobre $\mathcal{L}(\boldsymbol{1})$ indican el valor de $\mu$, pero aquí sin traducción a ningún estadístico, pues la escala de $\boldsymbol{x}$ es arbitraria—. Fíjate también en las líneas grises discontinuas que unen $\hat\beta_1\boldsymbol{1}$ (naranja) y $\hat\beta_2\boldsymbol{x}$ (siena) con la punta de $\boldsymbol{\mathop{\widehat{y}}}$: son los dos lados del paralelogramo cuya suma reconstruye el ajuste.



In [1]:
x, y = ejemplos['canonico']
fig = figura_ajuste_R3(x, y, rango=5, titulo="Ajuste de y sobre L(1,x)")
fig.show()

Junto a la figura 3D, su diagrama de dispersión gemelo: los mismos tres puntos $(x_i,y_i)$, ahora en el plano, con la recta de mínimos cuadrados y los residuos como segmentos verticales discontinuos (la misma información que el segmento gris de la figura 3D, vista “desde otro ángulo”, literalmente).



In [1]:
diagrama_dispersion(x, y, mostrar_residuos=True,
                     titulo="Diagrama de dispersión y recta ajustada")

#### Comprobación numérica: las dos ortogonalidades de la lección 7



In [1]:
yhat, ehat, beta0, beta1 = ajuste_mco_simple(y, x)
print(f"beta0_hat = {beta0:.4f}   beta1_hat = {beta1:.4f}")
print(f"suma de residuos (debe ser 0):      {ehat.sum():.10f}")
print(f"producto residuos . x (debe ser 0): {np.dot(ehat, x):.10f}")

#### Actividad



Cambia al ejemplo `fuerte` o `debil` (o crea el tuyo) y vuelve a ejecutar las tres celdas anteriores. Observa cómo cambia la *longitud* del segmento gris (el residuo) y compárala con lo que verás en el diagrama de dispersión: cuanto más “pegados” estén los puntos a la recta, más corto será ese segmento.



## Parte 2: los vectores en desviaciones, en $\mathcal{L}(\boldsymbol{1})^\perp$



Recordemos la identidad guardada en la lección 7: el vector en desviaciones del ajuste coincide exactamente con el de $\hat\beta_2\boldsymbol{x}$,
$$
\boldsymbol{\mathop{\widehat{y}}} - \boldsymbol{\mathop{\overline{y}}} \;=\; \hat\beta_2\big(\boldsymbol{x}-\boldsymbol{\mathop{\overline{x}}}\big).
$$
Como los tres vectores en desviaciones ($\boldsymbol{x}-\boldsymbol{\mathop{\overline{x}}}$, $\boldsymbol{y}-\boldsymbol{\mathop{\overline{y}}}$, $\boldsymbol{\mathop{\widehat{y}}}-\boldsymbol{\mathop{\overline{y}}}$) tienen media cero, los tres viven dentro del mismo plano $\mathcal{L}(\boldsymbol{1})^\perp$ que usamos en el primer cuaderno. Podemos, por tanto, trasladarlos todos al origen y dibujarlos juntos.

Añadimos también, en azul, la propia recta $\mathcal{L}(\boldsymbol{1})$: aunque no pertenece a este plano —es, de hecho, *perpendicular* a él—, sirve como referencia para orientar la figura. Si giras esta figura hasta que $\mathcal{L}(\boldsymbol{1})$ apunte directamente hacia ti (saliendo de la pantalla), estarás mirando el plano $\mathcal{L}(\boldsymbol{1})^\perp$ "de frente"; intenta poner la figura de la Parte 1 en esa misma orientación y comprueba que ambas coinciden.



In [1]:
def figura_desviaciones_R3(x, y, rango=4, titulo=""):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    yhat, ehat, beta0, beta1 = ajuste_mco_simple(y, x)
    ex, mu_x = vector_en_desviaciones(x)
    ey, mu_y = vector_en_desviaciones(y)
    eyhat, mu_yhat = vector_en_desviaciones(yhat)

    fig = go.Figure()
    recta_L1(fig, rango=rango)
    plano_ortogonal(fig, rango=rango)
    agrega_vector(fig, ex, color='black', nombre='x - x_barra', dash='dash')
    agrega_vector(fig, ey, color='yellowgreen', nombre='y - y_barra')
    agrega_vector(fig, eyhat, color='darkviolet',
                  nombre='yhat - y_barra', dash='dash')

    marcas_desviacion(fig, np.zeros(3), ex, n, color='black', etiqueta='sigma_x')
    marcas_desviacion(fig, np.zeros(3), ey, n, color='yellowgreen', etiqueta='sigma_y')
    marcas_desviacion(fig, np.zeros(3), eyhat, n, color='darkviolet', etiqueta='sigma_yhat')

    fondo = dict(showbackground=True, backgroundcolor='rgb(235,235,245)',
                 gridcolor='white', dtick=1, range=[-rango, rango])
    fig.update_layout(
        scene=dict(xaxis=fondo, yaxis=fondo, zaxis=fondo, aspectmode='cube'),
        title=titulo, width=800, height=700,
        margin=dict(l=0, r=0, b=0, t=40))
    return fig

x, y = ejemplos['canonico']
fig2 = figura_desviaciones_R3(x, y, rango=4,
                               titulo="Vectores en desviaciones en L(1)^perp")
fig2.show()

Rota la figura hasta ver los tres vectores “de perfil”: los vectores negro discontinuo ($\boldsymbol{x}-\boldsymbol{\mathop{\overline{x}}}$) y violeta discontinuo ($\boldsymbol{\mathop{\widehat{y}}}-\boldsymbol{\mathop{\overline{y}}}$) deben verse sobre la misma recta (misma dirección, o dirección opuesta si $\hat\beta_2<0$); el verde ($\boldsymbol{y}-\boldsymbol{\mathop{\overline{y}}}$), en cambio, forma con ellos el ángulo $\theta$ que protagonizará el próximo cuaderno.



#### Comprobación numérica



In [1]:
ex, _ = vector_en_desviaciones(x)
eyhat, _ = vector_en_desviaciones(yhat)
print("yhat - y_barra:       ", np.round(eyhat, 4))
print("beta1 * (x - x_barra):", np.round(beta1 * ex, 4))
print("¿son el mismo vector? ", np.allclose(eyhat, beta1 * ex))

#### Actividad



Repite esta figura con el ejemplo `debil`. Fíjate en el ángulo entre el vector verde y el violeta/negro: debería verse mucho más abierto que con el ejemplo `canonico`. Este ángulo, y su coseno al cuadrado, serán exactamente el $R^2$ del próximo cuaderno.



## Parte 3: ¿qué necesitamos exactamente de $\boldsymbol{u}$?



Como en la lección 7, imaginemos que los datos $\boldsymbol{y}$ se generaron según $\boldsymbol{y}=\beta_1\boldsymbol{1}+\beta_2\boldsymbol{x}+\boldsymbol{u}$, con $\beta_1,\beta_2$ conocidos por nosotros (el diseñador del experimento) pero *no* por MCO, que solo ve $\boldsymbol{y}$ y $\boldsymbol{x}$. *Advertencia*: $\boldsymbol{u}$ es un vector usado para generar $\boldsymbol{y}$, y no debe confundirse con el vector de residuos $\boldsymbol{\mathop{\widehat{e}}}$ que MCO calcula como diferencia entre $\boldsymbol{y}$ y el ajuste.

En $\mathbb{R}^3$ con dos regresores, $\mathcal{L}(\boldsymbol{1},\boldsymbol{x})^\perp$ es una única *recta* (recordemos la introducción): solo hay una dirección posible para que $\boldsymbol{u}$ quede *enteramente* fuera del plano. Vamos a comparar cinco vectores $\boldsymbol{u}$ distintos y ver, en cada caso, si MCO recupera $\beta_1,\beta_2$ exactamente.



In [1]:
def experimento_u(x, beta0, beta1, u, nombre=""):
    """Genera y = beta0*1 + beta1*x + u, ajusta MCO y devuelve un dict
    con los resultados, listo para comparar con los valores verdaderos."""
    x = np.asarray(x, dtype=float)
    u = np.asarray(u, dtype=float)
    uno = np.ones_like(x)
    y = beta0 * uno + beta1 * x + u
    yhat, ehat, beta0_hat, beta1_hat = ajuste_mco_simple(y, x)
    return dict(nombre=nombre, u=u, y=y, yhat=yhat, ehat=ehat,
                beta0_hat=beta0_hat, beta1_hat=beta1_hat,
                suma_u=float(u.sum()), producto_u_x=float(np.dot(u, x)))

def resumen(res):
    print(f"{res['nombre']:42s} "
          f"sum(u)={res['suma_u']:+6.2f}  u.x={res['producto_u_x']:+6.2f}   "
          f"beta0_hat={res['beta0_hat']:+7.4f}  beta1_hat={res['beta1_hat']:+7.4f}")

Fijamos $\boldsymbol{x}=(1,2,3)$ y los valores verdaderos $\beta_1=2,\beta1=3$ (el mismo ejemplo, en espíritu, que el de la lección 7). El truco para encontrar la *única* dirección ortogonal a la vez a $\boldsymbol{1}$ y a $\boldsymbol{x}$ es el producto vectorial: $\boldsymbol{1}\times\boldsymbol{x}$ es, por definición, perpendicular a ambos.



In [1]:
x = np.array([1., 2., 3.])
beta0, beta1 = 2., 3.
uno = np.ones_like(x)

direccion_perp = np.cross(uno, x)
print("dirección ortogonal a 1 y a x (1 x x):", direccion_perp)
print("¿ortogonal a 1?", np.isclose(direccion_perp.sum(), 0))
print("¿ortogonal a x?", np.isclose(np.dot(direccion_perp, x), 0))

Con esa dirección construimos los cinco casos:



In [1]:
u_a = np.zeros(3)                     # (a) sin perturbación
u_b = 2 * direccion_perp              # (b) u ⟂ 1 y u ⟂ x
u_c = np.array([-1., 0., 1.])         # (c) media(u)=0, pero u no ⟂ x
u_d = np.array([1., 1., -1.])         # (d) u ⟂ x, pero media(u) != 0
u_e = np.array([2., -1., 3.])         # (e) u no ⟂ 1 ni ⟂ x

casos = [
    ("(a) u = 0", u_a),
    ("(b) u ortogonal a 1 y a x", u_b),
    ("(c) media(u)=0, pero u no ortogonal a x", u_c),
    ("(d) u ortogonal a x, pero media(u) != 0", u_d),
    ("(e) u no ortogonal a 1 ni a x", u_e),
]

print(f"Valores verdaderos: beta0 = {beta0}, beta1 = {beta1}\n")
resultados = []
for nombre, u in casos:
    res = experimento_u(x, beta0, beta1, u, nombre)
    resultados.append(res)
    resumen(res)

#### Dos figuras, para contrastar



Visualizamos el caso más favorable, (b), y el más desfavorable, (e).



In [1]:
res_b = resultados[1]
fig_b = figura_ajuste_R3(x, res_b['y'], rango=6,
    titulo="Caso (b): u en L(1,x)^perp -- recuperacion exacta de beta0,beta1")
fig_b.show()

In [1]:
res_e = resultados[4]
fig_e = figura_ajuste_R3(x, res_e['y'], rango=6,
    titulo="Caso (e): u sin ninguna ortogonalidad")
fig_e.show()

#### Interpretación



El caso (a) es trivial: sin perturbación, no hay nada que estimar. El caso (b) recupera $\beta_1,\beta_2$ *exactamente*: es el análogo, en $\mathbb{R}^3$, del ejemplo “con truco” de la lección 7 —allí, con $n=5$, había todo un subespacio de dimensión 3 donde podía vivir $\boldsymbol{u}$ sin estropear el ajuste; aquí, con $n=3$ y dos regresores, esa libertad se ha reducido a una única recta—.

Los casos (c) y (d) son los más instructivos. Cada uno satisface *solo una* de las dos condiciones de ortogonalidad —(c) tiene media cero pero no es ortogonal a $\boldsymbol{x}$; (d) es ortogonal a $\boldsymbol{x}$ pero tiene media distinta de cero— y en ambos $\hat\beta_1,\hat\beta_2$ se alejan de los valores verdaderos. Esto confirma, con números, lo que ya anticipaba la lección 6: que el residuo de una proyección es ortogonal a *todo* el plano exige ortogonalidad a *cada uno* de sus generadores *simultáneamente*. Para que el residuo recupere el componente $\boldsymbol{u}$, es necesario que $\boldsymbol{u}$ sea ortogonal a ambos regresores; satisfacer solo una de las dos condiciones no basta. El caso (e), sin ninguna de las dos, es en general el más alejado de todos. Solo cuando el residuo coincide con $\boldsymbol{u}$, los betas estimados coinciden con los verdaderos valores usados para quenerar $\boldsymbol{y}$.

Guardemos esta idea: solo cuando $\boldsymbol{u}$ vive enteramente en la (única) dirección ortogonal al plano de los regresores, MCO recupera los coeficientes generadores con exactitud. Esta es, en miniatura, la intuición geométrica detrás de la condición $E[\boldsymbol{U}\mid\boldsymbol{X}]=\boldsymbol{0}$ que exigiremos a la perturbación poblacional en la lección 9.



#### Actividad



Elige tu propio vector $\boldsymbol{u}$ (o modifica alguno de los cinco anteriores) y comprueba en qué categoría cae —calculando $\sum u_i$ y $\boldsymbol{u}\cdot\boldsymbol{x}$— antes de ejecutar `experimento_u`. Intenta encontrar un $\boldsymbol{u}\neq\boldsymbol{0}$ que, sin estar exactamente en la dirección $\boldsymbol{1}\times\boldsymbol{x}$, recupere aproximadamente bien los coeficientes: ¿qué tiene de especial, comparado con los que sí lo consiguen exactamente?



## Para seguir explorando



-   El parámetro `rango` controla el tamaño de la escena; aumenta su valor si tus vectores tienen componentes grandes (como en el caso (b) o (e) de la Parte 3).
-   El diccionario `ejemplos` admite cualquier par $(\boldsymbol{x},\boldsymbol{y})$ nuevo: solo tienen que ser vectores de $\mathbb{R}^3$ con $\boldsymbol{x}$ no constante.
-   El próximo cuaderno (Notebook 3) retoma exactamente la figura de la Parte 2 —los vectores en desviaciones dentro de $\mathcal{L}(\boldsymbol{1})^\perp$— para construir el triángulo rectángulo de la lección 8 y visualizar $R^2=\cos^2\theta$.
-   La sesión de laboratorio que sigue a la lección 7 comprobará las mismas dos ortogonalidades, pero con datos reales.

